In [0]:
SELECT
    s.provider_id,
    p.vehicle_type,
    COUNT(*) AS stations,
    COUNT_IF(s.hours_since_reported > 24 * 30) AS older_than_30_days,
    ROUND(MEDIAN(s.hours_since_reported), 1) AS median_hours_since_reported,
    COUNT_IF(NOT (s.is_installed AND s.is_renting AND s.is_returning)) AS not_in_service
FROM dev.bikeshare.silver_station_status AS s
LEFT JOIN dev.bikeshare.bronze_providers AS p
    ON s.provider_id = p.provider_id
GROUP BY ALL
ORDER BY older_than_30_days DESC;

SELECT
    provider_id,
    provider_id IN ('velospot', 'nextbike_ch', 'nextbike_ms')
        OR provider_id LIKE 'donkey_%' AS is_bike_system,
    COUNT(*) AS stations,
    COUNT(capacity) AS with_capacity,
    ROUND(100.0 * COUNT(capacity) / COUNT(*), 0) AS pct_with_capacity,
    MIN(capacity) AS min_capacity,
    MAX(capacity) AS max_capacity
FROM dev.bikeshare.silver_station_information
WHERE __END_AT IS NULL
GROUP BY ALL
ORDER BY is_bike_system DESC, stations DESC;

SELECT provider_id, station_id, name, lat, lon
FROM dev.bikeshare.silver_station_information
WHERE __END_AT IS NULL
  AND NOT (lat BETWEEN 45.7 AND 47.9 AND lon BETWEEN 5.9 AND 10.6);

-- check scd type 1
SELECT 'providers' AS t, COUNT(*) AS n_rows FROM dev.bikeshare.silver_providers
UNION ALL
SELECT 'regions', COUNT(*) FROM dev.bikeshare.silver_system_regions;

-- check scd type 2
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(__END_AT IS NULL) AS current_rows,
    COUNT_IF(__END_AT IS NOT NULL) AS closed_versions
FROM dev.bikeshare.silver_station_information;

SELECT provider_id, station_id, name, lat, lon, capacity, region_id, __START_AT, __END_AT
FROM dev.bikeshare.silver_station_information
WHERE (provider_id, station_id) IN (
    SELECT provider_id, station_id
    FROM dev.bikeshare.silver_station_information
    WHERE __END_AT IS NOT NULL
)
ORDER BY __START_AT;

/*
DESCRIBE dev.bikeshare.bronze_station_status;
#DESCRIBE dev.bikeshare.bronze_station_information;
#DESCRIBE dev.bikeshare.bronze_providers;
*/